# Multi-Year ERA5 Capacity-Factor Variability — British Columbia

**Author:** Md Eliasinul Islam
**Affiliation:** Delta E+ Lab, Simon Fraser University
**Reviewer task:** **M4** — *Multi-year ERA5 CF variability* (addresses **R2.M2** "insufficient uncertainty treatment" and Editor comment **EC15**).
**Deliverables for Section 4.3:** per-cluster / per-region **CV statistics**, **cluster-rank Spearman rho** across weather years, and a **supply-curve envelope** band.

---

## What this notebook does

For each weather year it ingests the BASELINE run outputs
(`resource_options_<tech>_<region>_<year>.csv` and the matching `_timeseries.csv`)
and quantifies how the wind and solar resource - and the resulting screening-level
supply curves - vary from one ERA5 weather year to the next.

## Input folder convention (your model run)

```
<RESULTS_ROOT>/
  BASELINE_2014_<RUNDATE>/clusters/
      resource_options_solar_British Columbia_2014.csv
      resource_options_solar_British Columbia_2014_timeseries.csv
      resource_options_wind_British Columbia_2014.csv
      resource_options_wind_British Columbia_2014_timeseries.csv
  BASELINE_2015_<RUNDATE>/ ...
  BASELINE_2024_<RUNDATE>/ ...
```

The notebook **auto-discovers** every `BASELINE_<year>_<rundate>/clusters` folder, so you
only set the root once. Run dates may differ per year - discovery handles that.

## Methodological note - read before interpreting

K-means clustering is re-run **independently for each weather year** (features = screening
LCOE proxy + capacity), so **cluster IDs are not stable across years**. Three consequences:

1. *Robust, ID-independent metrics* (province- and regional-district-level capacity-weighted
   CF, supply-curve envelopes, accessible-GW spread) are the primary, defensible results.
2. *Cluster-rank Spearman rho* requires **re-matching clusters by spatial centroid** to a
   reference year before ranks are comparable. The notebook does this with a KD-tree and
   reports the match quality; treat rho as indicative, not exact.
3. Regional districts **are** stable, so the **region-level** Spearman rho is the cleaner
   rank-stability statistic and is reported alongside the cluster-level one.


---
## 0 - Configuration  - *edit only this cell*

In [ ]:
from pathlib import Path

# Where the per-year run folders live. Point at the directory CONTAINING the
# BASELINE_<year>_<rundate> folders. RUN_DATE=None accepts any run date.
RESULTS_ROOT    = Path("../../results/Canada/BC")   # adjust if your runs land elsewhere
SCENARIO_PREFIX = "BASELINE"
RUN_DATE        = "20260603"          # e.g. "20260531" -> only that run stamp

REGION_NAME    = "British Columbia"   # appears in the CSV filenames
REGION_CODE    = "BC"
RESOURCE_TYPES = ["wind", "solar"]

LCOE_CEILING   = 90       # USD/MWh feasibility ceiling for "accessible capacity"
WEATHER_YEARS  = None     # None = every year discovered; or e.g. list(range(2014, 2025))

OUT_DIR   = Path("./vis/multiyear_era5")   # figures + tidy CSVs written here
FIG_DPI   = 300
SAVE_TIFF = True

MPLSTYLE_CANDIDATES = [
    Path("../../codebase/RES/visual_styles/elsevier.mplstyle"),
    Path("../../codebase/visual_styles/elsevier.mplstyle"),
]

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results root :", RESULTS_ROOT.resolve(), "(exists:", RESULTS_ROOT.exists(), ")")
print("Output dir   :", OUT_DIR.resolve())


---
## 1 - Imports & plotting style

In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from scipy.spatial import cKDTree
    from scipy.stats import spearmanr
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False
    print("scipy not available -> Spearman/centroid matching will be skipped.")

for _sty in MPLSTYLE_CANDIDATES:
    if _sty.exists():
        plt.style.use(str(_sty)); print("Style applied:", _sty); break
else:
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": False})
    print("House style not found - using readable defaults.")

TECH_COLORS = {"wind": "#1565C0", "solar": "#E64A19"}
plt.rcParams["savefig.bbox"] = "tight"


---
## 2 - Discover run folders

Parses every `<PREFIX>_<year>_<rundate>` folder, records the weather year and run date, and checks which of the four expected CSVs are present. This manifest is your first sanity check after a run syncs.

In [ ]:
_folder_re = re.compile(rf"^{re.escape(SCENARIO_PREFIX)}_(?P<year>\d{{4}})_(?P<rundate>\d+)$")

def discover_runs(root: Path) -> pd.DataFrame:
    rows = []
    if not root.exists():
        print(f"!! RESULTS_ROOT does not exist yet: {root}")
        return pd.DataFrame()
    for d in sorted(root.iterdir()):
        if not d.is_dir():
            continue
        m = _folder_re.match(d.name)
        if not m:
            continue
        year = int(m.group("year")); rundate = m.group("rundate")
        if RUN_DATE is not None and rundate != RUN_DATE:
            continue
        if WEATHER_YEARS is not None and year not in WEATHER_YEARS:
            continue
        clusters = d / "clusters"
        rec = {"year": year, "rundate": rundate, "folder": d.name, "clusters_dir": clusters}
        for tech in RESOURCE_TYPES:
            stem = f"resource_options_{tech}_{REGION_NAME}_{year}"
            rec[f"{tech}_clusters"]   = (clusters / f"{stem}.csv").exists()
            rec[f"{tech}_timeseries"] = (clusters / f"{stem}_timeseries.csv").exists()
        rows.append(rec)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.sort_values("year").reset_index(drop=True)
    if df["year"].duplicated().any():           # keep latest run date per year
        df = (df.sort_values(["year", "rundate"])
                .drop_duplicates("year", keep="last").reset_index(drop=True))
    return df

manifest = discover_runs(RESULTS_ROOT)
if manifest.empty:
    print("No run folders found yet. Re-run this cell once the model output has synced.")
else:
    print(f"Discovered {len(manifest)} weather-year run(s): "
          f"{manifest['year'].min()}-{manifest['year'].max()}")
_disp = ["year", "rundate"] + [c for c in manifest.columns
                               if c.endswith(("_clusters", "_timeseries"))]
manifest[_disp] if not manifest.empty else manifest


---
## 3 - Loaders with column auto-detection

The exact column labels (`lcoe_wind` vs `lcoe`, `wind_CF_mean` vs `CF_mean`, `x/y` vs `lon/lat`) depend on the framework version. These loaders map whatever is present to a canonical schema: `capacity_mw`, `lcoe`, `cf_mean`, `dist_km`, `x`, `y`, `region`. Absent fields become `NaN` and dependent analyses skip gracefully.

In [ ]:
def _pick(cols, candidates, contains=None):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    if contains:
        for c in cols:
            cl = c.lower()
            if all(tok in cl for tok in contains):
                return c
    return None

def load_clusters(tech, year, clusters_dir):
    p = clusters_dir / f"resource_options_{tech}_{REGION_NAME}_{year}.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p, index_col=0); cols = list(df.columns)
    canon = pd.DataFrame(index=df.index); canon.index.name = "cluster_id"
    cap  = _pick(cols, ["potential_capacity", f"potential_capacity_{tech}"], ["potential", "capacit"])
    lcoe = _pick(cols, ["lcoe", f"lcoe_{tech}", "lcoe_variant", "screening_lcoe"], ["lcoe"])
    cf   = _pick(cols, [f"{tech}_CF_mean", "CF_mean", "cf_mean", "capacity_factor"], ["cf", "mean"])
    dist = _pick(cols, ["nearest_station_distance_km", "dist_km"], ["distance", "km"])
    xcol = _pick(cols, ["x", "lon", "longitude"], ["lon"])
    ycol = _pick(cols, ["y", "lat", "latitude"], ["lat"])
    reg  = _pick(cols, ["Region", "region", "regional_district", "sub_national_unit"], ["region"])
    canon["capacity_mw"] = pd.to_numeric(df[cap],  errors="coerce") if cap  else np.nan
    canon["lcoe"]        = pd.to_numeric(df[lcoe], errors="coerce") if lcoe else np.nan
    canon["cf_mean"]     = pd.to_numeric(df[cf],   errors="coerce") if cf   else np.nan
    canon["dist_km"]     = pd.to_numeric(df[dist], errors="coerce") if dist else np.nan
    canon["x"]           = pd.to_numeric(df[xcol], errors="coerce") if xcol else np.nan
    canon["y"]           = pd.to_numeric(df[ycol], errors="coerce") if ycol else np.nan
    canon["region"]      = df[reg].astype(str) if reg else "ALL"
    canon["tech"] = tech; canon["year"] = year
    return canon

def load_timeseries(tech, year, clusters_dir):
    p = clusters_dir / f"resource_options_{tech}_{REGION_NAME}_{year}_timeseries.csv"
    if not p.exists():
        return None
    ts = pd.read_csv(p, index_col=0)
    ts.index = pd.to_datetime(ts.index, errors="coerce", utc=True)
    ts = ts[~ts.index.isna()]
    ts.columns = [str(c) for c in ts.columns]
    return ts.apply(pd.to_numeric, errors="coerce")

if not manifest.empty:
    r0 = manifest.iloc[0]
    _ex = load_clusters(RESOURCE_TYPES[0], int(r0["year"]), r0["clusters_dir"])
    if _ex is not None:
        print("Resolved canonical columns with data:",
              [c for c in _ex.columns if _ex[c].notna().any()])


---
## 4 - Build master tables

`clusters_long` stacks every cluster from every (tech, year). The province capacity-weighted annual-mean CF is computed two independent ways - from `cf_mean` and from the hourly time series - as a cross-check.

In [ ]:
def cap_weighted(values, weights):
    v = np.asarray(values, float); w = np.asarray(weights, float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    return np.nan if not m.any() else np.average(v[m], weights=w[m])

clusters_long, province_rows, ts_store = [], [], {}

for _, r in manifest.iterrows() if not manifest.empty else []:
    yr = int(r["year"]); cdir = r["clusters_dir"]
    for tech in RESOURCE_TYPES:
        cl = load_clusters(tech, yr, cdir)
        if cl is None:
            continue
        clusters_long.append(cl)
        cf_from_mean = cap_weighted(cl["cf_mean"], cl["capacity_mw"])
        ts = load_timeseries(tech, yr, cdir)
        cf_from_ts = np.nan
        if ts is not None:
            idx_str = cl.index.astype(str)
            common = [c for c in ts.columns if c in set(idx_str)]
            if common:
                cap_map = dict(zip(idx_str, cl["capacity_mw"].astype(float).values))
                w = np.array([cap_map[c] for c in common], float)
                w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
                if w.sum() > 0:
                    prov = ts[common].mul(w, axis=1).sum(axis=1) / w.sum()
                    cf_from_ts = float(prov.mean()); ts_store[(tech, yr)] = prov
        province_rows.append({
            "tech": tech, "year": yr, "n_clusters": len(cl),
            "total_capacity_gw": cl["capacity_mw"].sum() / 1e3,
            "cf_capwt_from_mean": cf_from_mean, "cf_capwt_from_ts": cf_from_ts,
            "accessible_gw": (cl.loc[cl["lcoe"] <= LCOE_CEILING, "capacity_mw"].sum() / 1e3
                              if cl["lcoe"].notna().any() else np.nan)})

clusters_long = pd.concat(clusters_long) if clusters_long else pd.DataFrame()
province = (pd.DataFrame(province_rows).sort_values(["tech", "year"]).reset_index(drop=True)
           if province_rows else pd.DataFrame())
province


---
## 5 - Interannual CF variability (headline statistics)

The coefficient of variation (CV = SD / mean) of the province capacity-weighted annual-mean CF is the single most reviewer-relevant number: how much the *resource itself* moves between weather years, before any economic assumption.

In [ ]:
def cv(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return np.nan if (x.size == 0 or x.mean() == 0) else x.std(ddof=1) / x.mean()

interannual = []
if not province.empty:
    for tech in RESOURCE_TYPES:
        sub = province[province["tech"] == tech]
        if sub.empty:
            continue
        cf = sub["cf_capwt_from_ts"].fillna(sub["cf_capwt_from_mean"])
        acc = sub["accessible_gw"]
        interannual.append({
            "tech": tech, "n_years": int(cf.notna().sum()),
            "CF_mean": cf.mean(), "CF_sd": cf.std(ddof=1), "CF_CV_pct": 100 * cv(cf),
            "CF_min": cf.min(), "CF_max": cf.max(),
            "CF_min_year": int(sub.loc[cf.idxmin(), "year"]) if cf.notna().any() else None,
            "CF_max_year": int(sub.loc[cf.idxmax(), "year"]) if cf.notna().any() else None,
            "accessibleGW_mean": acc.mean(), "accessibleGW_sd": acc.std(ddof=1),
            "accessibleGW_CV_pct": 100 * cv(acc),
            "accessibleGW_min": acc.min(), "accessibleGW_max": acc.max()})
interannual = pd.DataFrame(interannual).round(4)
if not interannual.empty:
    interannual.to_csv(OUT_DIR / "interannual_cf_stats.csv", index=False)
    print("Saved -> interannual_cf_stats.csv")
interannual


### Figure M4-1 - Distribution of cluster annual-mean CF by weather year

In [ ]:
clusters_long

In [ ]:
if not clusters_long.empty:
    fig, axes = plt.subplots(1, len(RESOURCE_TYPES),
                             figsize=(6.2 * len(RESOURCE_TYPES), 4.4), squeeze=False)
    for ax, tech in zip(axes[0], RESOURCE_TYPES):
        sub = clusters_long[clusters_long["tech"] == tech]
        yrs = sorted(sub["year"].unique())
        data = [sub.loc[sub["year"] == y, "cf_mean"].dropna().values for y in yrs]
        if not any(len(d) for d in data):
            ax.set_visible(False); continue
        bp = ax.boxplot(data, positions=range(len(yrs)), widths=0.6,
                        patch_artist=True, showfliers=False)
        for box in bp["boxes"]:
            box.set(facecolor=TECH_COLORS[tech], alpha=0.35, edgecolor=TECH_COLORS[tech])
        for med in bp["medians"]:
            med.set(color="black", linewidth=1.2)
        prov = province[province["tech"] == tech].set_index("year")
        cw = []
        for y in yrs:
            v = prov.loc[y, "cf_capwt_from_ts"] if y in prov.index else np.nan
            if not np.isfinite(v) and y in prov.index:
                v = prov.loc[y, "cf_capwt_from_mean"]
            cw.append(v)
        ax.plot(range(len(yrs)), cw, "o-", color=TECH_COLORS[tech], lw=2,
                mfc="white", label="capacity-weighted mean")
        ax.set_xticks(range(len(yrs))); ax.set_xticklabels(yrs, rotation=45, ha="right")
        ax.set_title(tech.title(), fontweight="bold")
        ax.set_ylabel("Annual-mean capacity factor"); ax.set_xlabel("Weather year")
        ax.legend(fontsize=8, frameon=False, loc="lower right")
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig_M4-1_cf_distribution_by_year.png", dpi=FIG_DPI)
    # if SAVE_TIFF: fig.savefig(OUT_DIR / "fig_M4-1_cf_distribution_by_year.tiff", dpi=FIG_DPI)
    plt.show()
else:
    print("No cluster data loaded yet.")


---
## 6 - Supply-curve envelope across weather years

Each year yields one screening-level supply curve (clusters sorted by LCOE proxy, cumulative capacity on x). Overlaying all years and shading the min-max envelope shows how stable the feasible space is to weather-year choice - the direct visual answer to R2.M2.

In [ ]:
def supply_curve(df):
    d = df.dropna(subset=["lcoe", "capacity_mw"]).sort_values("lcoe")
    return d["capacity_mw"].cumsum().values / 1e3, d["lcoe"].values

if not clusters_long.empty and clusters_long["lcoe"].notna().any():
    fig, axes = plt.subplots(1, len(RESOURCE_TYPES),
                             figsize=(6.2 * len(RESOURCE_TYPES), 4.6), squeeze=False)
    for ax, tech in zip(axes[0], RESOURCE_TYPES):
        sub = clusters_long[clusters_long["tech"] == tech]
        yrs = sorted(sub["year"].unique())
        xmax = max((supply_curve(sub[sub.year == y])[0].max()
                    for y in yrs if len(sub[sub.year == y])), default=1)
        grid = np.linspace(0, xmax, 400); stacks = []
        for y in yrs:
            cg, lc = supply_curve(sub[sub["year"] == y])
            if len(cg) == 0:
                continue
            ax.step(cg, lc, where="post", color=TECH_COLORS[tech], alpha=0.25, lw=0.9)
            stacks.append(np.interp(grid, cg, lc, left=np.nan, right=np.nan))
        if stacks:
            M = np.vstack(stacks)
            ax.fill_between(grid, np.nanmin(M, 0), np.nanmax(M, 0),
                            color=TECH_COLORS[tech], alpha=0.15, label="min-max envelope")
            ax.plot(grid, np.nanmedian(M, 0), color=TECH_COLORS[tech], lw=2.2,
                    label="median year")
        ax.axhline(LCOE_CEILING, color="black", ls=":", lw=1.1,
                   label=f"LCOE ceiling ({LCOE_CEILING})")
        ax.set_title(f"{tech.title()} - supply-curve envelope", fontweight="bold")
        ax.set_xlabel("Cumulative developable potential (GW)")
        ax.set_ylabel("Screening-level LCOE proxy (USD/MWh)")
        ax.set_ylim(0, LCOE_CEILING * 2.2); ax.set_xlim(left=0)
        ax.legend(fontsize=8, frameon=False, loc="upper left")
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig_M4-2_supply_curve_envelope.png", dpi=FIG_DPI)
    # if SAVE_TIFF: fig.savefig(OUT_DIR / "fig_M4-2_supply_curve_envelope.tiff", dpi=FIG_DPI)
    plt.show()
else:
    print("No LCOE data loaded yet -> supply-curve section skipped.")


---
## 7 - Seasonal CF profile and its interannual envelope

Monthly capacity-weighted CF, averaged across weather years, with a +/-1 SD ribbon showing year-to-year spread per month. This separates systematic seasonality (robust) from interannual noise (the uncertainty the reviewer asked about).

In [ ]:
if ts_store:
    fig, axes = plt.subplots(1, len(RESOURCE_TYPES),
                             figsize=(6.2 * len(RESOURCE_TYPES), 4.2), squeeze=False)
    fig.suptitle(f"Seasonal profiles for Weather Years - {REGION_NAME}", fontweight="bold", fontsize=14)
    seasonal_out = []
    for ax, tech in zip(axes[0], RESOURCE_TYPES):
        series = {y: s for (t, y), s in ts_store.items() if t == tech}
        if not series:
            ax.set_visible(False); continue
        monthly = {y: s.groupby(s.index.month).mean().reindex(range(1, 13))
                   for y, s in series.items()}
        M = pd.DataFrame(monthly)
        mean_m, sd_m = M.mean(axis=1), M.std(axis=1, ddof=1)
        months = np.arange(1, 13)
        ax.fill_between(months, mean_m - sd_m, mean_m + sd_m,
                        color=TECH_COLORS[tech], alpha=0.2, label="+/-1 SD across years")
        ax.plot(months, mean_m, "o-", color=TECH_COLORS[tech], lw=2, label="multi-year mean")
        ax.set_xticks(months)
        ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
        ax.set_title(f"{tech.title()} - seasonal profile", fontweight="bold")
        ax.set_ylabel("Capacity factor"); ax.set_xlabel("Month")
        ax.legend(fontsize=8, frameon=False)
        ax.spines[["top", "right"]].set_visible(False)
        tmp = M.copy(); tmp.columns = [f"{tech}_{c}" for c in tmp.columns]
        seasonal_out.append(tmp)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fig_M4-3_seasonal_profile.svg", dpi=FIG_DPI)
    # if SAVE_TIFF: fig.savefig(OUT_DIR / "fig_M4-3_seasonal_profile.tiff", dpi=FIG_DPI)
    plt.show()
    if seasonal_out:
        pd.concat(seasonal_out, axis=1).to_csv(OUT_DIR / "seasonal_monthly_cf.csv")
        print("Saved -> seasonal_monthly_cf.csv")
else:
    print("No timeseries loaded yet -> seasonal section skipped.")


---
## 8 - Rank stability - Spearman rho across weather years

**Region level (primary):** regional districts are stable, so we rank them by capacity-weighted CF each year and correlate every year against a reference year. **Cluster level (indicative):** clusters are re-matched to the reference year by nearest centroid (KD-tree on `x,y`) before ranking; the median match distance is reported so you can judge match quality. High rho means the ordering of good sites is robust to weather year even if absolute CF shifts.

In [ ]:
def region_capwt_cf(sub):
    g = sub.dropna(subset=["cf_mean", "capacity_mw"])
    return g.groupby("region").apply(lambda d: cap_weighted(d["cf_mean"], d["capacity_mw"]))

rank_results, rank_df = {}, pd.DataFrame()
if _HAVE_SCIPY and not clusters_long.empty:
    for tech in RESOURCE_TYPES:
        sub = clusters_long[clusters_long["tech"] == tech]
        yrs = sorted(sub["year"].unique())
        if len(yrs) < 2:
            continue
        ref = yrs[len(yrs) // 2]
        ref_reg = region_capwt_cf(sub[sub["year"] == ref])
        reg_rho = {}
        for y in yrs:
            cur = region_capwt_cf(sub[sub["year"] == y])
            j = pd.concat([ref_reg.rename("ref"), cur.rename("cur")], axis=1).dropna()
            reg_rho[y] = spearmanr(j["ref"], j["cur"]).correlation if len(j) > 2 else np.nan
        clu_rho, match_d = {}, {}
        ref_df = sub[sub["year"] == ref].dropna(subset=["x", "y", "cf_mean"])
        if len(ref_df) > 2:
            tree = cKDTree(ref_df[["x", "y"]].values)
            for y in yrs:
                cur = sub[sub["year"] == y].dropna(subset=["x", "y", "cf_mean"])
                if len(cur) < 3:
                    clu_rho[y] = np.nan; continue
                dist, idx = tree.query(cur[["x", "y"]].values, k=1)
                clu_rho[y] = spearmanr(ref_df["cf_mean"].values[idx], cur["cf_mean"].values).correlation
                match_d[y] = float(np.median(dist))
        rank_results[tech] = {"ref_year": ref, "region_rho": reg_rho,
                              "cluster_rho": clu_rho, "median_match_dist": match_d}
    rows = []
    for tech, r in rank_results.items():
        for y in sorted(r["region_rho"]):
            rows.append({"tech": tech, "ref_year": r["ref_year"], "year": y,
                         "region_spearman": r["region_rho"].get(y),
                         "cluster_spearman": r["cluster_rho"].get(y),
                         "median_match_dist": r["median_match_dist"].get(y)})
    rank_df = pd.DataFrame(rows).round(3)
    if not rank_df.empty:
        rank_df.to_csv(OUT_DIR / "rank_stability_spearman.csv", index=False)
        print("Saved -> rank_stability_spearman.csv")
    for tech, r in rank_results.items():
        rr = [v for k, v in r["region_rho"].items() if k != r["ref_year"] and v == v]
        cc = [v for k, v in r["cluster_rho"].items() if k != r["ref_year"] and v == v]
        if rr and cc:
            print(f"{tech:>5}: ref={r['ref_year']}  region rho min={min(rr):.3f} "
                  f"mean={np.mean(rr):.3f} | cluster rho min={min(cc):.3f} mean={np.mean(cc):.3f}")
else:
    print("scipy missing or no data -> rank stability skipped.")
rank_df


---
## 9 - Paste-ready summary for Section 4.3 / Response M4

A compact text block assembling the numbers reviewers asked for. Copy the printed template into the manuscript PLACEHOLDER and the Response-to-Reviewers M4 row, then adapt the wording.

In [ ]:
def fmt(x, p=1):
    return "--" if (x is None or (isinstance(x, float) and not np.isfinite(x))) else f"{x:.{p}f}"

print("=" * 78)
print("MULTI-YEAR ERA5 CF VARIABILITY - SUMMARY  (M4 / R2.M2 / EC15)")
print("=" * 78)
if not manifest.empty:
    print(f"Weather years analysed: {manifest['year'].min()}-{manifest['year'].max()} "
          f"(n={manifest['year'].nunique()})\n")
for _, row in interannual.iterrows():
    t = row["tech"]
    print(f"[{t.upper()}]")
    print(f"  Capacity-weighted annual-mean CF: {fmt(row['CF_mean'],3)} "
          f"(range {fmt(row['CF_min'],3)}-{fmt(row['CF_max'],3)}; CV {fmt(row['CF_CV_pct'])}%); "
          f"lowest {row['CF_min_year']}, highest {row['CF_max_year']}.")
    print(f"  Accessible capacity at LCOE <= {LCOE_CEILING}: {fmt(row['accessibleGW_mean'],2)} GW mean "
          f"(range {fmt(row['accessibleGW_min'],2)}-{fmt(row['accessibleGW_max'],2)} GW; "
          f"CV {fmt(row['accessibleGW_CV_pct'])}%).")
    if not rank_df.empty:
        rr = rank_df[rank_df.tech == t]
        reg = rr["region_spearman"].dropna(); clu = rr["cluster_spearman"].dropna()
        if len(reg): print(f"  Region-rank Spearman rho vs reference: min {fmt(reg.min(),3)}, mean {fmt(reg.mean(),3)}.")
        if len(clu): print(f"  Cluster-rank Spearman rho (centroid-matched): min {fmt(clu.min(),3)}, mean {fmt(clu.mean(),3)}.")
    print()
print("All figures + CSVs written to:", OUT_DIR.resolve())
print("=" * 78)
